# 03 · Análisis estadístico

Nivel de significación α = 0,05. Con muestras muy grandes casi todo sale «significativo», así que **se reporta siempre el tamaño de efecto**.

**Hipótesis (definidas antes de analizar):**
- **H1.** La proporción de lesiones graves/mortales depende de la lluvia.
- **H2.** El nº de accidentes diarios difiere entre días con y sin lluvia.
- **H3.** El nº de accidentes diarios difiere según el día de la semana.
- **H4.** Existe correlación entre temperatura media y accidentes diarios.
- **H5.** La gravedad se asocia con franja horaria, alcohol, sexo y tipo de vehículo, controlando por lluvia.

In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from src.config import *
from src.stats_utils import *
from src.features import collapse_top
df = pd.read_csv(FINAL_CSV, parse_dates=["fecha"])
pers = df[(df["gravedad"] != "desconocido") & df["lluvia"].notna()].copy()
acc = df.drop_duplicates("num_expediente")
daily = acc.groupby("fecha").agg(accidentes=("num_expediente", "size"), tmed=("meteo_tmed", "first"),
        prec=("meteo_prec", "first"), lluvia=("lluvia", "first"), dia_semana=("dia_semana", "first"),
        es_festivo=("es_festivo", "first")).dropna(subset=["lluvia", "tmed"]).reset_index()
print(len(pers), len(daily))

141905 2082


## Normalidad y supuestos
Los conteos diarios son discretos y asimétricos → se usan contrastes no paramétricos.

In [2]:
print("Shapiro (muestra 5000) accidentes/día:", stats.shapiro(daily["accidentes"].sample(min(5000, len(daily)), random_state=1)))
print("Media:", daily["accidentes"].mean().round(2), "| Varianza:", daily["accidentes"].var().round(2), "(varianza > media → sobredispersión)")

Shapiro (muestra 5000) accidentes/día: ShapiroResult(statistic=np.float64(0.9939196436220434), pvalue=np.float64(1.4795715765236545e-07))
Media: 52.37 | Varianza: 292.5 (varianza > media → sobredispersión)


## H1 · Lluvia × gravedad (chi-cuadrado)

In [3]:
r = chi2_report(pers, "lluvia", "gravedad")
print({k: round(v, 4) for k, v in r.items() if k != "tabla"})
display(pd.crosstab(pers["lluvia"], pers["gravedad"], normalize="index").mul(100).round(2))
for l in (0, 1):
    s = pers[pers["lluvia"] == l]
    p, lo, hi = prop_ci(int(s["es_grave"].sum()), len(s))
    print(f"lluvia={l}: % grave/mortal = {p*100:.2f}%  IC95% [{lo*100:.2f}, {hi*100:.2f}]")

{'chi2': np.float64(19.8462), 'p': np.float64(0.0005), 'gl': 4, 'V_cramer': 0.0118, 'min_esperada': 35.2279}


gravedad,grave,ileso,leve,moderado,mortal
lluvia,,,,,
0.0,2.14,55.76,34.79,7.19,0.13
1.0,2.01,54.92,35.98,7.01,0.08


lluvia=0: % grave/mortal = 2.27%  IC95% [2.18, 2.35]
lluvia=1: % grave/mortal = 2.09%  IC95% [1.94, 2.26]


**Interpretación:** el chi-cuadrado rechaza la independencia (χ² = 19,8; gl = 4; p = 0,0005), pero el efecto es despreciable: V de Cramér = 0,012 (< 0,1). Los porcentajes de lesiones graves o mortales son casi iguales: 2,27 % sin lluvia (IC 95 % 2,18-2,35) y 2,09 % con lluvia (1,94-2,26). Con tantas observaciones, p < 0,05 no implica relevancia práctica: la lluvia no cambia de forma apreciable la gravedad.  
**Nota:** las personas de un mismo accidente no son independientes; se confirma en H5 con errores robustos por accidente.

## H2 · Accidentes/día con vs sin lluvia (Mann-Whitney)

In [4]:
x = daily.loc[daily["lluvia"] == 1, "accidentes"]; y = daily.loc[daily["lluvia"] == 0, "accidentes"]
print("Mediana lluvia:", x.median(), "| sin lluvia:", y.median(), "| n:", len(x), len(y))
print(mannwhitney_report(x, y))

Mediana lluvia: 57.0 | sin lluvia: 52.0 | n: 425 1657
{'U': np.float64(399050.5), 'p': np.float64(2.1757346781542113e-05), 'rank_biserial': np.float64(-0.1333039866519934)}


## H3 · Día de la semana (Kruskal-Wallis + post-hoc)

In [5]:
order = ["lunes","martes","miercoles","jueves","viernes","sabado","domingo"]
groups = [daily.loc[daily["dia_semana"] == d, "accidentes"].values for d in order]
print(kruskal_report(groups))
# post-hoc: Mann-Whitney por pares con corrección de Bonferroni
import itertools
pairs = list(itertools.combinations(order, 2)); res = []
for a, b in pairs:
    p = stats.mannwhitneyu(daily.loc[daily.dia_semana==a,"accidentes"], daily.loc[daily.dia_semana==b,"accidentes"]).pvalue
    res.append((a, b, p, min(p*len(pairs), 1)))
post = pd.DataFrame(res, columns=["a","b","p","p_bonferroni"])
post[post["p_bonferroni"] < 0.05].sort_values("p_bonferroni").head(15)

{'H': np.float64(289.59043973412537), 'p': np.float64(1.389031233572225e-59), 'epsilon2': np.float64(0.13667009143813272)}


,a,b,p,p_bonferroni
19,viernes,domingo,3.944617e-39,8.283696e-38
17,jueves,domingo,3.371822e-30,7.080826e-29
10,martes,domingo,3.224115e-29,6.770641e-28
14,miercoles,domingo,8.707485e-27,1.828572e-25
18,viernes,sabado,1.157903e-20,2.431596e-19
5,lunes,domingo,2.164097e-18,4.544603e-17
3,lunes,viernes,8.272765e-14,1.737281e-12
16,jueves,sabado,1.431908e-11,3.007007e-10
9,martes,sabado,1.446351e-09,3.037336e-08
20,sabado,domingo,1.471675e-08,3.090517e-07


## H4 · Correlación temperatura / precipitación con accidentes (Spearman)

In [6]:
for v in ["tmed", "prec"]:
    rho, p = stats.spearmanr(daily[v], daily["accidentes"])
    print(f"{v}: rho={rho:.3f}  p={p:.4g}")

tmed: rho=-0.164  p=4.93e-14
prec: rho=0.092  p=2.724e-05


## Modelo de conteos: accidentes/día (Binomial Negativa)
El modelo controla a la vez por calendario y meteorología.

In [7]:
daily["mes"] = daily["fecha"].dt.month; daily["anio"] = daily["fecha"].dt.year
nb = smf.glm("accidentes ~ lluvia + tmed + C(dia_semana) + es_festivo + C(anio) + C(mes)", data=daily,
             family=__import__("statsmodels.api", fromlist=["x"]).families.NegativeBinomial(alpha=0.1)).fit()
irr = np.exp(nb.params).rename("IRR"); ci = np.exp(nb.conf_int()).rename(columns={0: "IC2.5", 1: "IC97.5"})
pd.concat([irr, ci, nb.pvalues.rename("p")], axis=1).loc[["lluvia", "tmed", "es_festivo"]].round(4)

,IRR,IC2.5,IC97.5,p
lluvia,1.0448,1.0056,1.0855,0.0248
tmed,1.0043,0.9993,1.0093,0.0943
es_festivo,0.6197,0.5680,0.6761,0.0000


IRR > 1 = más accidentes esperados por unidad de aumento (manteniendo lo demás constante). Con `alpha` = 0,1 (valor fijo) la lluvia daba IRR = 1,045 (p = 0,025); con `alpha` estimado a partir de los datos (0,03, celda al final del notebook) resulta IRR = 1,046 (IC 95 % 1,021-1,073; p = 0,0003).

## H5 · Regresión logística: ¿qué factores se asocian a una lesión grave?

In [8]:
m = pers.copy()
m["tipo_vehiculo_c"] = collapse_top(m["tipo_vehiculo"], 6)
m = m[m["sexo"].isin(["hombre", "mujer"])]
m["alcohol"] = m["positiva_alcohol"].fillna(0)
fit = smf.logit("es_grave ~ C(lluvia) + C(franja_horaria, Treatment('mañana')) + C(sexo) + alcohol + es_fin_semana + C(tipo_vehiculo_c)",
                data=m).fit(disp=0, cov_type="cluster", cov_kwds={"groups": pd.factorize(m["num_expediente"])[0]})
out = pd.concat([np.exp(fit.params).rename("OR"), np.exp(fit.conf_int()).rename(columns={0:"IC2.5",1:"IC97.5"}), fit.pvalues.rename("p")], axis=1).round(4)
print("Pseudo R2:", round(fit.prsquared, 4), "| n =", int(fit.nobs))
out

Pseudo R2: 0.0724 | n = 141465


,OR,IC2.5,IC97.5,p
Intercept,0.0301,0.0245,0.0370,0.0000
C(lluvia)[T.1.0],0.9400,0.8548,1.0337,0.2017
"C(franja_horaria, Treatment('mañana'))[T.madrugada]",1.6105,1.3899,1.8661,0.0000
"C(franja_horaria, Treatment('mañana'))[T.noche]",0.9572,0.8554,1.0712,0.4463
"C(franja_horaria, Treatment('mañana'))[T.tarde]",0.7488,0.6813,0.8231,0.0000
C(sexo)[T.mujer],1.1221,1.0357,1.2158,0.0048
C(tipo_vehiculo_c)[T.bicicleta],1.5740,1.2275,2.0183,0.0003
C(tipo_vehiculo_c)[T.furgoneta],0.4817,0.3620,0.6412,0.0000
C(tipo_vehiculo_c)[T.motocicleta > 125cc],3.0350,2.4646,3.7375,0.0000
C(tipo_vehiculo_c)[T.motocicleta hasta 125cc],1.9152,1.5573,2.3553,0.0000


In [9]:
from sklearn.metrics import roc_auc_score  # opcional: pip install scikit-learn
print("AUC:", roc_auc_score(m["es_grave"], fit.predict(m)))

AUC: 0.7243459648413444


## Conclusiones estadísticas (α = 0,05)

**H1 · Lluvia y gravedad.** La independencia se rechaza formalmente (χ² = 19,8; p = 0,0005), pero el efecto es despreciable (V de Cramér = 0,012): 2,27 % de personas con lesión grave o mortal sin lluvia y 2,09 % con lluvia. En la regresión logística con conductores, la lluvia no aumenta la gravedad (OR = 0,88; IC 95 % 0,78–0,99; p = 0,03). Lectura práctica: no hay evidencia de que la lluvia empeore la gravedad; el ligero descenso es débil y no debe interpretarse como causal.

**H2 · Accidentes por día con y sin lluvia.** Mediana de 57 con lluvia frente a 52 sin lluvia (425 y 1.657 días); Mann-Whitney p < 0,001 con efecto pequeño (rank-biserial = −0,13; el signo negativo indica más accidentes en días de lluvia). Controlando calendario y temperatura (binomial negativa, α estimado = 0,03), los días de lluvia tienen un 4,6 % más de accidentes (IRR = 1,046; IC 95 % 1,021–1,073).

**H3 · Día de la semana.** Kruskal-Wallis H = 289,6; p < 0,001; ε² = 0,137 (efecto moderado). El viernes tiene más accidentes (mediana 65) y el domingo menos (42); el post-hoc con Bonferroni separa los fines de semana de los días laborables.

**H4 · Temperatura y precipitación.** Correlaciones débiles: ρ = −0,16 con la temperatura y ρ = 0,09 con la precipitación. La correlación negativa con la temperatura refleja sobre todo la estacionalidad: al controlar mes y año, el IRR por grado es 1,004 (aprox. +4 % cada 10 °C; p = 0,011). El día festivo tiene un 38 % menos de accidentes (IRR = 0,62).

**H5 · Factores asociados a lesión grave (solo conductores; referencia: turismo).** Aumentan las probabilidades: moto de más de 125 cc (OR = 25,1), moto de hasta 125 cc (16,8), bicicleta (14,2), madrugada frente a mañana (2,2), noche (1,2), fin de semana (1,2) y alcohol positivo (1,34; IC 95 % 1,05–1,70). Las mujeres conductoras tienen menos probabilidad (OR = 0,73). Pseudo R² = 0,16.

**Limitaciones.**
- Una sola estación meteorológica (Retiro) para toda la ciudad, y la lluvia diaria no captura la hora exacta.
- Solo accidentes con víctimas registrados por la Policía Municipal, y sin datos de exposición al tráfico.
- Se excluyen unos 110 días (5 %) sin dato de lluvia o temperatura, y el 43,6 % de las personas sin lesividad registrada en H1 y H5; si esa ausencia no es aleatoria, hay sesgo.
- La prueba de alcohol no se hace a todos los implicados (posible sesgo de selección).
- Varias personas comparten accidente: H5 usa errores robustos por accidente, pero el χ² de H1 no lo corrige.
- Son asociaciones, no causalidad.
- La estación no registra viento desde mediados de 2020, por lo que no se usa.

In [10]:
dias = acc.groupby("fecha").agg(tmed=("meteo_tmed", "first"), lluvia=("lluvia", "first"))
print("Días con accidentes:", len(dias))
print("Sin temperatura:", dias["tmed"].isna().sum(), "| sin lluvia:", dias["lluvia"].isna().sum())
print()
print(daily.groupby("dia_semana")["accidentes"].median().reindex(order))
print()
print(pd.crosstab(m["alcohol"], m["gravedad"], normalize="index").round(3))
print()
print(m["tipo_vehiculo_c"].value_counts())

Días con accidentes: 2192
Sin temperatura: 20 | sin lluvia: 91

dia_semana
lunes        52.0
martes       57.0
miercoles    55.5
jueves       58.0
viernes      65.0
sabado       48.0
domingo      42.0
Name: accidentes, dtype: float64

gravedad  grave  ileso   leve  moderado  mortal
alcohol                                        
0.0       0.021  0.552  0.354     0.071   0.001
1.0       0.026  0.638  0.256     0.080   0.000

tipo_vehiculo_c
turismo                    88606
motocicleta hasta 125cc    14175
otros                      12807
motocicleta > 125cc         9451
furgoneta                   8154
autobus                     4464
bicicleta                   3808
Name: count, dtype: int64


In [11]:
import statsmodels.api as sm

# 1) alpha estimado con los datos
f = "accidentes ~ lluvia + tmed + C(dia_semana) + es_festivo + C(anio) + C(mes)"
pois = smf.glm(f, data=daily, family=sm.families.Poisson()).fit()
mu = np.asarray(pois.mu)
aux = ((daily["accidentes"].values - mu) ** 2 - mu) / mu
alpha = max(float(np.asarray(sm.OLS(aux, mu).fit().params)[0]), 1e-6)
print("alpha estimado:", round(alpha, 4), "(antes: 0.1)")
nb2 = smf.glm(f, data=daily, family=sm.families.NegativeBinomial(alpha=alpha)).fit()
ci = np.exp(nb2.conf_int()).rename(columns={0: "IC2.5", 1: "IC97.5"})
display(pd.concat([np.exp(nb2.params).rename("IRR"), ci, nb2.pvalues.rename("p")], axis=1)
        .loc[["lluvia", "tmed", "es_festivo"]].round(4))

# 2) y 3) regresión logística solo con conductores, turismo como referencia
cond = m[m["tipo_persona"] == "conductor"].copy()
fit2 = smf.logit("es_grave ~ C(lluvia) + C(franja_horaria, Treatment('mañana')) + C(sexo) + alcohol"
                 " + es_fin_semana + C(tipo_vehiculo_c, Treatment('turismo'))", data=cond
                 ).fit(disp=0, cov_type="cluster", cov_kwds={"groups": pd.factorize(cond["num_expediente"])[0]})
ci2 = np.exp(fit2.conf_int()).rename(columns={0: "IC2.5", 1: "IC97.5"})
print("n =", int(fit2.nobs), "| Pseudo R2:", round(fit2.prsquared, 4))
display(pd.concat([np.exp(fit2.params).rename("OR"), ci2, fit2.pvalues.rename("p")], axis=1).round(4))

alpha estimado: 0.0305 (antes: 0.1)


,IRR,IC2.5,IC97.5,p
lluvia,1.0464,1.0209,1.0725,0.0003
tmed,1.0042,1.0009,1.0075,0.0114
es_festivo,0.6200,0.5846,0.6577,0.0000


n = 106109 | Pseudo R2: 0.1609


,OR,IC2.5,IC97.5,p
Intercept,0.0031,0.0026,0.0036,0.0000
C(lluvia)[T.1.0],0.8768,0.7774,0.9890,0.0324
"C(franja_horaria, Treatment('mañana'))[T.madrugada]",2.2247,1.8512,2.6735,0.0000
"C(franja_horaria, Treatment('mañana'))[T.noche]",1.2062,1.0467,1.3901,0.0096
"C(franja_horaria, Treatment('mañana'))[T.tarde]",0.9035,0.7990,1.0216,0.1055
C(sexo)[T.mujer],0.7265,0.6307,0.8369,0.0000
"C(tipo_vehiculo_c, Treatment('turismo'))[T.autobus]",0.2947,0.0731,1.1887,0.0860
"C(tipo_vehiculo_c, Treatment('turismo'))[T.bicicleta]",14.1975,11.3933,17.6919,0.0000
"C(tipo_vehiculo_c, Treatment('turismo'))[T.furgoneta]",1.0528,0.6510,1.7024,0.8339
"C(tipo_vehiculo_c, Treatment('turismo'))[T.motocicleta > 125cc]",25.0977,21.1648,29.7614,0.0000


In [12]:
cols = ["fecha", "es_festivo", "meteo_tmed", "meteo_tmin", "meteo_tmax", "meteo_prec", "lluvia"]
acc.drop_duplicates("fecha")[cols].to_csv(FINAL_CSV.parent / "diario_meteo_festivos.csv", index=False)